# Building the API layer

The goal here is to make requests to the API, receive a JSON, and return Python dictionaries. A thin wrapper around HTTP requests.

Something like this:

def get_ride(ride_id): \
    response = requests.get(...) \
    return response.json() \

To include the parent folder in the path:

In [38]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [39]:
BASE_URL="https://global.api.flixbus.com"

In [40]:
import requests
from config import STATIONS
from config import API_KEY #import the sensitive API_KEY from the config file, which in turn loads it from the .env file
from datetime import datetime, timedelta, timezone

import json

Reload the config.py if you changed something

In [ ]:
import importlib
import config
importlib.reload(config) #reload the config module to ensure we have the latest changes, especially

In [ ]:
# Something that the copilot gave me, but let's keep our project for simple. We added manually the station ids in the config.py file, so we don't need to search for them. 
#def search_station(query):
#    url = f"{BASE_URL}/search/stations"
#    params = {
#        "query": query,
#        "locale": "en-US",
#        "key": API_KEY
#    }
#    response = requests.get(url, params=params)#\
#    return response.json()
    

We inserted the station IDs manually of just a couple of stations in Italy. 

Format the time and date. We have to take good care of this, because we will maybe grab data every minute/5 minutes or whatever. They should be dynamic. 
So:
1) Let's call the function to take data every 5 minutes
2) Take data in a window between now - 30 minutes to now + 4 hours or something.
We want to take data on the temporal evolution and stuff later on maybe. For now, maybe, the most important thing is just the final delay information. 

In [ ]:
#Flixbus time format is ISO 8601, i.e. this one below, so we need to convert our format to this one
#time_from = "2026-05-07T17:48:00.000Z"
#time_to = "2026-05-07T20:18:00.000Z"


#Function for formatting the time for the flixbus API and for the filenames
def format_time_for_flixbus_api(time):
    timeflix = time.isoformat(timespec="minutes") + ":00.000Z" #get the current time in ISO 8601 format, which is the format required by the FLIXBUS API
    timeflix=timeflix.replace(":", "%3A") #replace the ":" with "%3A" to make it URL safe, as required by the FLIXBUS API
    
    timefile = time.isoformat(timespec="minutes").replace(":", "_").replace("-", "_") #replace the ":" and "-" with "_" to make it a valid filename
    return timeflix, timefile

time_now = datetime.now() #get the current time in ISO 8601 format, which is the format required by the FLIXBUS API
time_in_4_hours = time_now + timedelta(hours=4)
time_before_1_hour = time_now - timedelta(hours=1)
time_before_4_hours = time_now - timedelta(hours=4)
time_before_2and_half_hour = time_now - timedelta(hours=2.5)

time_now, time_now_file = format_time_for_flixbus_api(time_now)
time_before_4_hours, time_before_4_hours_file = format_time_for_flixbus_api(time_before_4_hours)
time_in_4_hours, time_in_4_hours_file = format_time_for_flixbus_api(time_in_4_hours)
time_before_2and_half_hour, time_before_2and_half_hour_file = format_time_for_flixbus_api(time_before_2and_half_hour) #flixbus seems to have only these links
time_before_1_hour, time_before_1_hour_file = format_time_for_flixbus_api(time_before_1_hour)

#print("The correct format \n" + f"2026-05-07T17:48:00.000Z".replace(":", "%3A")) #check if the formatting is correct, it should print "2026-05-07T17%3A48%3A00.000Z"

#print(time_now)
#print(time_now_file)

The correct format 
2026-05-07T17%3A48%3A00.000Z
2026-05-12T19%3A40%3A00.000Z
2026_05_12T19_40


In [84]:
def get_departures(station_id):
    #The URL looks like this: 
    #url=https://global.api.flixbus.com/gis/v2/timetable/dcbda963-9603-11e6-9066-549f350fcb0c/departures?from=2026-05-07T17%3A48%3A00.000Z&to=2026-05-07T20%3A18%3A00.000Z&apiKey=7781b8fa-07cf-4ab7-8b62-1f3178523ba0
    #the structure is base_url + /gis/v2/timetable/ + station_id + /departures?from= + from_time + &to= + to_time + &apiKey= + API_KEY

    url = f"{BASE_URL}/gis/v2/timetable/{station_id}/departures?from={time_before_4_hours}&to={time_in_4_hours}&apiKey={API_KEY}"
    print(url) #print the URL to check if it's correct
    response = requests.get(url)
    return response.json()

In [ ]:
station = "genoa_principe"

departs = get_departures(STATIONS[station]) # to get the station ID from the dictionary just write STATIONS[station], where station is the key of the station you want to get the departures for, in this case "genoa_principe"


with open("/home/armankorajac/flixbus_tracker/data/raw/{0}_departures_{1}_to_{2}.json".format(station, time_before_4_hours_file, time_in_4_hours_file), "w") as f:
    json.dump(departs, f, indent=4)

https://global.api.flixbus.com/gis/v2/timetable/dcc02ea8-9603-11e6-9066-549f350fcb0c/departures?from=2026-05-12T15%3A40%3A00.000Z&to=2026-05-12T23%3A40%3A00.000Z&apiKey=7781b8fa-07cf-4ab7-8b62-1f3178523ba0


Let's check how this looks like

In [123]:
departs.keys()


dict_keys(['rides', 'station'])

In [134]:
departs["rides"][5].keys()

dict_keys(['id', 'status', 'platform', 'line', 'location', 'calls', 'vehicle'])

In [129]:
departs["rides"][2]["id"]

'4b113bf2-6f4c-4a58-927f-47747ae4cc14'

In [136]:
departs['rides'][2]['platform']

Okay, we have a mechanism to get the raw .json files.

Now, how are the different JSON files connected? 
Note that Flixbus gives us 2 major datasets:
1) One is v2/timetable, which for every station gives us information about arrivals/departures from this city 
2) The other is v3/ride, which gives us information about RIDES and the stations inbetween that they transverse

From v2/timetable, which has the big array rides = [], we find the unique IDs of the RIDES themselves. 
Once we have these unique IDs, we can go to v3/ride and check anything we want about that specific ride (estimated time of arrival, real time of arrival/departure, whatever there is...).
Already the first file gives a lot of information about delays btw! 

What is to do then? 
I should:
1) Make this API call automatic, like every 5 minutes, and get the raw data
2) Should I then process the data and get info about delays at different timestamps? 
